# Deploy NLP model bằng FastAPI

Mục tiêu: dùng lại model phân loại tin tức tiếng Việt đã train, đưa model thành API và ghi nhận quá trình triển khai.

Model lấy từ `VNTC_text_classification_tensorflow.ipynb`. Không cần train lại, GPU hoặc API key.

Luồng: **Văn bản → POST /predict → kiểm tra input → SavedModel → chủ đề + điểm softmax → JSON**.

Notebook này chạy server trên **localhost**, phù hợp demo bằng Jupyter trên máy. Chưa publish lên cloud. Chọn kernel **Python 3.12** và chạy lần lượt. Khi xong, chạy cell dừng server ở cuối.

**Các file cần có**

Giữ notebook cùng thư mục với `nlp_fastapi_deployment/` và `vntc_tensorflow_outputs/`. Artifact cần có `labels.json` và toàn bộ `vntc_tensorflow_saved_model/` (gồm graph, assets, variables).

- `main.py`: schema input/output, nạp model và các endpoint.
- `client_demo.py`: gọi API từ terminal.
- `verify_deployment.py`: HTTP test, đo tốc độ, đánh giá một tập con và tạo báo cáo.
- `DEPLOYMENT_REPORT.md`: báo cáo có số đo thực tế.

SavedModel đã giữ preprocessing và vocabulary từ training. API nạp bằng `tf.saved_model.load`, gọi endpoint `serve`; không học lại vocabulary. [TensorFlow SavedModel](https://www.tensorflow.org/guide/saved_model).

In [ ]:
# Chỉ cần chạy khi kernel chưa có các dependency. Lần đầu tải TensorFlow có thể lâu.
%pip install -r nlp_fastapi_deployment/requirements.txt

**1. Kiểm tra đường dẫn**

Nếu Jupyter mở ở vị trí khác, sửa `WORKSPACE` thành đường dẫn chứa notebook và artifact. TensorFlow 2.16.1 trong requirements phù hợp Python 3.12; Python quá mới có thể không cài được version này.

In [1]:
from pathlib import Path
import json
import os
import socket
import subprocess
import sys
import time

import httpx
from IPython.display import Markdown, display

WORKSPACE = Path.cwd().resolve()
if WORKSPACE.name == "nlp_fastapi_deployment":
    WORKSPACE = WORKSPACE.parent
PROJECT = WORKSPACE / "nlp_fastapi_deployment"
MODEL_DIR = WORKSPACE / "vntc_tensorflow_outputs"
assert (PROJECT / "main.py").is_file(), "Sửa WORKSPACE để tìm được main.py."
assert (MODEL_DIR / "vntc_tensorflow_saved_model" / "saved_model.pb").is_file(), "Thiếu SavedModel."
assert (MODEL_DIR / "labels.json").is_file(), "Thiếu labels.json."
PORT = 8000
BASE_URL = f"http://127.0.0.1:{PORT}"
print("Python:", sys.version.split()[0])
print("Project:", PROJECT)
print("Model:", MODEL_DIR)
print("Số nhãn:", len(json.loads((MODEL_DIR / "labels.json").read_text(encoding="utf-8"))))

Python: 3.12.3
Project: D:\Sudo code 2026\nlp_fastapi_deployment
Model: D:\Sudo code 2026\vntc_tensorflow_outputs
Số nhãn: 10


**2. Khởi động FastAPI**

API nạp model một lần trong **lifespan**, warm-up rồi mới nhận request. Endpoint chạy inference dùng `def`, để FastAPI thực thi trong thread pool. [FastAPI Lifespan](https://fastapi.tiangolo.com/advanced/events/).

Cell này tạo process Uvicorn riêng, chờ `/health`, sau đó trả quyền điều khiển cho notebook. Nếu cổng bận, đổi `PORT` ở cell trước. Không dừng ứng dụng đang dùng cổng đó.

In [2]:
if globals().get("server_process") is not None and server_process.poll() is None:
    raise RuntimeError("Server notebook đã chạy. Dùng cell dừng server trước khi khởi động lại.")

with socket.socket() as probe:
    probe.bind(("127.0.0.1", PORT))

log_dir = PROJECT / "results"
log_dir.mkdir(exist_ok=True)
server_log = log_dir / "notebook_server.log"
environment = os.environ.copy()
environment["MODEL_DIR"] = str(MODEL_DIR)
environment["TF_CPP_MIN_LOG_LEVEL"] = "2"
environment["TF_NUM_INTRAOP_THREADS"] = "1"
environment["TF_NUM_INTEROP_THREADS"] = "1"
command = [sys.executable, "-m", "uvicorn", "main:app", "--app-dir", str(PROJECT),
           "--host", "127.0.0.1", "--port", str(PORT), "--workers", "1"]
with server_log.open("w", encoding="utf-8") as stream:
    server_process = subprocess.Popen(
        command, cwd=WORKSPACE, env=environment, stdout=stream, stderr=subprocess.STDOUT,
        creationflags=subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0,
    )

def stop_demo_server():
    process = globals().get("server_process")
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait(timeout=5)

started = time.perf_counter()
try:
    with httpx.Client(timeout=1, trust_env=False) as client:
        while True:
            if server_process.poll() is not None:
                raise RuntimeError(f"Server lỗi. Xem {server_log}")
            try:
                response = client.get(BASE_URL + "/health")
                if response.status_code == 200:
                    break
            except httpx.RequestError:
                pass
            if time.perf_counter() - started > 120:
                raise TimeoutError(f"Chờ server quá 120 giây. Xem {server_log}")
            time.sleep(0.25)
    print("Health:", response.json())
    display(Markdown(f"Thử API bằng [Swagger UI]({BASE_URL}/docs)."))
except BaseException:
    stop_demo_server()
    raise

Health: {'status': 'ok', 'model_id': 'vntc-tf-dd30f80c9902', 'num_classes': 10, 'startup_ms': 5068.084}


Thử API bằng [Swagger UI](http://127.0.0.1:8000/docs).

**3. Gọi API với một câu**

`text` phải là chuỗi có chữ/số, tối đa 20.000 ký tự. `top_k` là số nguyên 1–10, mặc định 3. Model đọc tối đa **400 token đầu** theo cấu hình khi train.

`score` là điểm softmax, chưa được hiệu chỉnh thành độ tin cậy. Model luôn chọn một trong 10 nhãn, kể cả với văn bản ngoài miền. Nhãn giữ nguyên tên không dấu của dataset.

In [3]:
question = "Đội tuyển bóng đá Việt Nam giành chiến thắng trong trận chung kết."
with httpx.Client(timeout=30, trust_env=False) as client:
    before = time.perf_counter()
    response = client.post(BASE_URL + "/predict", json={"text": question, "top_k": 3})
    response.raise_for_status()
    prediction = response.json()
    round_trip_ms = (time.perf_counter() - before) * 1000
print(json.dumps(prediction, ensure_ascii=False, indent=2))
print(f"HTTP round trip: {round_trip_ms:.2f} ms")

{
  "label": "The thao",
  "score": 1.0,
  "top_k": [
    {
      "label": "The thao",
      "score": 1.0
    },
    {
      "label": "Chinh tri Xa hoi",
      "score": 6.605476148696665e-12
    },
    {
      "label": "Phap luat",
      "score": 2.5493472486032634e-14
    }
  ],
  "inference_ms": 1.268,
  "model_id": "vntc-tf-dd30f80c9902"
}
HTTP round trip: 15.08 ms


**4. Thử vài câu và input lỗi**

Các câu ngắn dưới đây chỉ dùng minh họa. Accuracy cần tính trên tập dữ liệu có nhãn. Schema FastAPI kiểm tra JSON và trả HTTP 422 khi input không hợp lệ. [Request Body](https://fastapi.tiangolo.com/tutorial/body/).

In [4]:
examples = [
    "Đội tuyển bóng đá Việt Nam giành chiến thắng trong trận chung kết.",
    "Công ty công nghệ ra mắt phần mềm bảo mật và máy tính mới.",
    "Các bác sĩ khuyến cáo người dân tiêm vaccine để phòng bệnh.",
]
rows = []
with httpx.Client(base_url=BASE_URL, timeout=30, trust_env=False) as client:
    labels_response = client.get("/labels")
    labels_response.raise_for_status()
    print("Các nhãn:", labels_response.json()["labels"])
    for text in examples:
        response = client.post("/predict", json={"text": text, "top_k": 3})
        response.raise_for_status()
        item = response.json()
        rows.append(f"| {text} | {item['label']} | {item['score']:.4f} |")
    invalid = client.post("/predict", json={"text": "   ", "top_k": 3})
    assert invalid.status_code == 422
    print("Input chỉ khoảng trắng → HTTP", invalid.status_code)
display(Markdown("| Input | Dự đoán | Softmax |\n|---|---|---:|\n" + "\n".join(rows)))

Các nhãn: ['Chinh tri Xa hoi', 'Doi song', 'Khoa hoc', 'Kinh doanh', 'Phap luat', 'Suc khoe', 'The gioi', 'The thao', 'Van hoa', 'Vi tinh']
Input chỉ khoảng trắng → HTTP 422


| Input | Dự đoán | Softmax |
|---|---|---:|
| Đội tuyển bóng đá Việt Nam giành chiến thắng trong trận chung kết. | The thao | 1.0000 |
| Công ty công nghệ ra mắt phần mềm bảo mật và máy tính mới. | Vi tinh | 1.0000 |
| Các bác sĩ khuyến cáo người dân tiêm vaccine để phòng bệnh. | Suc khoe | 0.9286 |

**5. Đọc báo cáo triển khai**

`DEPLOYMENT_REPORT.md` ghi kết quả kiểm tra thực tế của script: endpoint, input lỗi, parity với SavedModel, độ trễ HTTP và đánh giá một tập con Test_Full nếu có.

Kết quả test **87,6% accuracy / macro-F1 0,8738 trên 8.000 mẫu** trong `vntc_tensorflow_outputs/metrics.json` thuộc lần train trước. Không lấy ba câu demo để tính lại accuracy.

Cell mặc định chỉ đọc báo cáo có sẵn. Đổi `REBUILD_REPORT = True` nếu muốn chạy lại toàn bộ; script dùng cổng 8765 và tự dừng server kiểm tra khi xong.

In [5]:
REBUILD_REPORT = False
if REBUILD_REPORT:
    # Chỉ chạy khi muốn có số đo mới; thời gian gồm nạp hai bản SavedModel để so sánh.
    subprocess.run([sys.executable, str(PROJECT / "verify_deployment.py")], cwd=WORKSPACE, check=True)

report = PROJECT / "DEPLOYMENT_REPORT.md"
if report.exists():
    display(Markdown(report.read_text(encoding="utf-8")))
else:
    print("Chưa có báo cáo. Đặt REBUILD_REPORT = True để tạo.")

# Báo cáo triển khai NLP model bằng FastAPI

Thời điểm chạy: **2026-10-28T17:51:12+07:00**. Báo cáo được tạo tự động bằng `verify_deployment.py` từ kết quả chạy thật.

**Mục tiêu và model**

Đưa model phân loại tin tức tiếng Việt từ `VNTC_text_classification_tensorflow.ipynb` thành HTTP API. Input là văn bản; output gồm chủ đề dự đoán, điểm softmax và top-k chủ đề.

Model sử dụng: `TextVectorization → Embedding → GlobalAveragePooling1D → Dense → Dropout → Softmax`, 10 nhãn. SavedModel giữ cả vocabulary và preprocessing; API gọi `tf.saved_model.load(...).serve(...)` và giữ đúng thứ tự `labels.json`. [Tài liệu SavedModel](https://www.tensorflow.org/guide/saved_model).

Artifact đang phục vụ: `vntc-tf-dd30f80c9902`. Mã này được tính từ SHA-256 của graph, trọng số, assets và labels; không tự chứng minh rằng labels là đúng về ngữ nghĩa.

**Quy trình thực hiện**

1. Dùng lại SavedModel đã huấn luyện trong `vntc_tensorflow_outputs`, không train lại.
2. Chuẩn bị môi trường Python và cài `requirements.txt`.
3. Viết schema input/output và ba endpoint `/health`, `/labels`, `/predict`.
4. Nạp model và warm-up một lần trong lifespan, trước khi nhận request. Thiếu model hoặc output sai thì startup thất bại. Cách quản lý này theo [FastAPI Lifespan](https://fastapi.tiangolo.com/advanced/events/).
5. Khởi động Uvicorn với một worker, bind `127.0.0.1`, gọi API qua HTTP thật.
6. Kiểm tra dữ liệu hợp lệ/không hợp lệ, so sánh với SavedModel, đo độ trễ và xuất báo cáo.

Luồng xử lý: `Client → HTTP POST /predict → validate → SavedModel (preprocessing + neural network) → xếp hạng → JSON`.

**Môi trường và phạm vi triển khai**

- Hệ điều hành: Windows-11-10.0.26200-SP0; Python 3.12.3.
- Thư viện: tensorflow 2.16.1, keras 3.15.1, numpy 1.26.4, fastapi 0.141.1, uvicorn 0.52.4, httpx 0.28.1, pydantic 2.14.1, starlette 1.10.0.
- Thiết bị TensorFlow nhìn thấy: `['CPU']`.
- Một worker, request tuần tự, TensorFlow intra/inter-op threads đặt bằng 1 trong script đo.
- Phạm vi: **API chạy cục bộ trên máy**. Chưa publish lên cloud hoặc tạo public URL.
- Script kiểm tra dừng server của chính nó sau khi đo. Muốn demo lâu hơn, khởi động server theo `README.md`.

**Kết quả khởi động và chức năng**

- Trạng thái `/health`: `ok`; số lớp: 10.
- Thời gian trong lifespan (import TensorFlow, đọc model, hash và warm-up): **6.02 giây**.
- Thời gian từ tạo process đến nhận được health: **7.65 giây**; gồm import server và khoảng polling.
- Các trường hợp HTTP dưới đây đều qua kiểm tra assertion, sử dụng model thật.

| Kiểm tra | HTTP status | Kết quả |
|---|---:|---|
| GET /health, model đã sẵn sàng | 200 | Pass |
| GET /labels đúng thứ tự training | 200 | Pass |
| GET /docs | 200 | Pass |
| GET /openapi.json | 200 | Pass |
| POST /predict hợp lệ | 200 | Pass |
| Thiếu text | 422 | Pass |
| Text rỗng | 422 | Pass |
| Chỉ khoảng trắng | 422 | Pass |
| Chỉ dấu câu | 422 | Pass |
| Text là số | 422 | Pass |
| Text null | 422 | Pass |
| Text quá 20000 ký tự | 422 | Pass |
| top_k bằng 0 | 422 | Pass |
| top_k bằng 11 | 422 | Pass |
| top_k là số thực | 422 | Pass |
| top_k là boolean | 422 | Pass |
| top_k là chuỗi | 422 | Pass |
| Trường JSON không hỗ trợ | 422 | Pass |
| JSON không hợp lệ | 422 | Pass |
| top_k mặc định bằng 3 | 200 | Pass |

Ngoài HTTP, đã kiểm tra nạp model từ thư mục thiếu artifact phải phát sinh `FileNotFoundError`.

**Ví dụ đầu ra thực tế**

| Input | Nhãn dự đoán | Điểm softmax |
|---|---|---:|
| Đội tuyển bóng đá Việt Nam giành chiến thắng trong trận chung kết. | The thao | 1.000000 |
| Công ty công nghệ ra mắt phần mềm bảo mật và máy tính mới. | Vi tinh | 1.000000 |
| Các bác sĩ khuyến cáo người dân tiêm vaccine để phòng bệnh. | Suc khoe | 0.928648 |

Các câu này chỉ dùng demo, không phải bộ benchmark độ chính xác. Điểm softmax không phải cam kết xác suất đúng; model luôn chọn một trong 10 chủ đề và chưa có cơ chế từ chối văn bản ngoài miền.

**Độ trễ**

Bỏ qua 3 request warm-up. Đo 30 request tuần tự, luân phiên ba câu demo, cùng một HTTP client và kết nối keep-alive.

| Phép đo | Trung bình (ms) | Median (ms) | P95 (ms) |
|---|---:|---:|---:|
| HTTP round trip tại localhost | 1.974 | 1.917 | 2.322 |
| `inference_ms` do API báo | 0.618 | 0.589 | 0.839 |

HTTP round trip tính từ trước khi gửi đến sau khi đọc JSON. `inference_ms` chỉ tính gọi model và xếp hạng; không tính chờ thread pool, validation hay truyền HTTP. Đây là phép đo demo trên máy hiện tại, không phải load test nhiều người dùng hoặc độ trễ cloud.

**Chất lượng mô hình**

Theo file `vntc_tensorflow_outputs/metrics.json` từ lần huấn luyện trước: accuracy **87.60%**, macro-F1 **0.8738**, trên **8000 mẫu test**. Các số này là kết quả lịch sử, không phải số đo mới từ deployment.

Script lấy **100 văn bản** từ `vntc_data/Test_Full`, 10 văn bản mỗi lớp, seed 42. Chỉ lấy văn bản không rỗng và không quá 20.000 ký tự. Số file bị bỏ qua trước khi chọn đủ mẫu: 0.

- Accuracy qua API: **87.00%**.
- Macro-F1 qua API: **0.8637**.
- Dự đoán API trùng với gọi trực tiếp SavedModel: **100/100**.
- Sai khác xác suất lớn nhất: `0`.

Đây là tập con kiểm tra triển khai; không phải chạy lại toàn bộ 8.000 mẫu test cũ. File `holdout_predictions.json` lưu đường dẫn tương đối, nhãn thật và dự đoán để kiểm tra lại.

**Điểm cần nhớ**

- API nhận tối đa 20.000 ký tự; TextVectorization trong model chỉ dùng 400 token đầu. Văn bản dài có thể mất thông tin ở cuối.
- Nhãn output giữ nguyên tên không dấu từ dataset. Model này dành cho tin tức tiếng Việt, không dùng để tóm tắt hoặc dịch.
- Input sai trả 422; lỗi inference trả 500. Validation thực hiện bằng schema theo [FastAPI Request Body](https://fastapi.tiangolo.com/tutorial/body/).
- Khi đưa lên server công khai cần cấu hình HTTPS, xác thực, giới hạn request và tài nguyên phù hợp. Bài này chỉ triển khai demo cục bộ.

**File bằng chứng**

- `results/20261028T105106_560543Z/verification.json`: cấu hình, versions, kiểm tra và số đo tổng hợp.
- `results/20261028T105106_560543Z/holdout_predictions.json`: dự đoán có nhãn và so sánh trực tiếp/API, nếu có test data.
- `results/20261028T105106_560543Z/server.log`: log Uvicorn.
- `results/20261028T105106_560543Z/latencies.json`: từng lần đo độ trễ.

Hướng dẫn chạy lại: xem `README.md`. Tham khảo cách chạy Uvicorn trong [FastAPI deployment](https://fastapi.tiangolo.com/deployment/manually/).


**6. Dừng server khi demo xong**

Cell dưới chỉ dừng process do notebook này tạo. Sau khi dừng, `/docs` và `/predict` sẽ không truy cập được nữa; chạy lại cell khởi động nếu cần demo tiếp.

In [6]:
stop_demo_server()
print("Đã dừng server của notebook.")

Đã dừng server của notebook.


**Nhận xét**

- Training tạo model; deployment đưa model thành dịch vụ nhận request và trả dự đoán.
- Cần đóng gói trọng số, preprocessing và nhãn cùng nhau để giữ đúng hành vi lúc train.
- Lifespan tránh nạp lại model ở mỗi request; validation giúp báo lỗi rõ khi client gửi sai dữ liệu.
- Thời gian `inference_ms` chỉ đo gọi model và xếp hạng. HTTP round trip còn gồm xử lý request và truyền dữ liệu.
- Đây là deployment cục bộ với một worker. Chưa kiểm tra tải đồng thời, chưa publish cloud.

Xem thêm `nlp_fastapi_deployment/README.md` để chạy bằng terminal. [FastAPI deployment](https://fastapi.tiangolo.com/deployment/manually/).